# NB12 — Within-Species Subclade Analysis

**Goal**: Test H7 — do plant-adaptation genes segregate by subclade within key species?

Approach (revised for feasibility):
1. Select top plant-associated species with >=20 genomes
2. Use `phylogenetic_tree_distance_pairs` (22.6M rows) to define subclades
3. Cluster genomes within species by phylogenetic distance (agglomerative clustering)
4. Map isolation source and host plant onto subclades
5. Test: are plant-derived genomes concentrated in specific subclades?

**Note**: Avoids 1B-row `gene` table. Uses phylogenetic distance as the natural
subclade definition rather than gene-content clustering.

**Outputs**:
- `data/subclade_og_enrichment.csv`
- `data/subclade_host_mapping.csv`
- `data/species_subclade_definitions.csv`
- `figures/subclade_analysis.png`

In [1]:
import pandas as pd
import numpy as np
import os, warnings
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
warnings.filterwarnings('ignore')

DATA = '../data'
FIG  = '../figures'
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

spark = get_spark_session()
print('Spark session ready')

# Load Phase 1/2 data
sp_comp = pd.read_csv(f'{DATA}/species_compartment.csv')
ge = pd.read_csv(f'{DATA}/genome_environment.csv')
refined = pd.read_csv(f'{DATA}/species_cohort_refined.csv')
host_genomes = pd.read_csv(f'{DATA}/genome_host_species.csv')
markers_v2 = pd.read_csv(f'{DATA}/species_marker_matrix_v2.csv')

print(f'Genomes: {len(ge):,}')
print(f'Species: {len(sp_comp):,}')
print(f'Plant-associated species: {sp_comp["is_plant_associated"].sum():,}')

Spark session ready


Genomes: 293,059
Species: 26,511
Plant-associated species: 1,136


## 1. Select Target Species

Choose plant-associated species with >=20 genomes for subclade analysis.
Start with top 5 to validate approach before scaling.

In [2]:
# Count genomes per plant-associated species
plant_species = sp_comp[sp_comp['is_plant_associated'] == True]['gtdb_species_clade_id']
genome_counts = ge[ge['gtdb_species_clade_id'].isin(plant_species)].groupby(
    'gtdb_species_clade_id').size().sort_values(ascending=False)

# Filter to >= 20 genomes
candidates = genome_counts[genome_counts >= 20]
print(f'Plant-associated species with >= 20 genomes: {len(candidates)}')

# Select top 5 for initial analysis
target_species = candidates.head(5).index.tolist()
print(f'\nTarget species for subclade analysis:')
for sp in target_species:
    genus = sp_comp[sp_comp['gtdb_species_clade_id'] == sp]['genus'].iloc[0]
    n = genome_counts[sp]
    cohort = refined[refined['gtdb_species_clade_id'] == sp]['cohort_refined'].iloc[0] \
             if sp in refined['gtdb_species_clade_id'].values else 'unknown'
    # Count plant vs non-plant genomes
    sp_ge = ge[ge['gtdb_species_clade_id'] == sp]
    n_plant = sp_ge['is_plant_associated'].sum()
    print(f'  {sp[:60]}: {n} genomes ({n_plant} plant, {n-n_plant} non-plant) [{genus}, {cohort}]')

Plant-associated species with >= 20 genomes: 65

Target species for subclade analysis:
  s__Xanthomonas_euvesicatoria--RS_GCF_020880415.1: 332 genomes (169 plant, 163 non-plant) [g__Xanthomonas, dual-nature]
  s__Pseudomonas_E_avellanae--RS_GCF_000444135.1: 319 genomes (68 plant, 251 non-plant) [g__Pseudomonas_E, dual-nature]
  s__Sinorhizobium_meliloti--RS_GCF_017876815.1: 241 genomes (191 plant, 50 non-plant) [g__Sinorhizobium, dual-nature]
  s__Pseudomonas_E_amygdali--RS_GCF_002699855.1: 239 genomes (32 plant, 207 non-plant) [g__Pseudomonas_E, dual-nature]
  s__Rhizobium_laguerreae--RS_GCF_002008165.1: 175 genomes (139 plant, 36 non-plant) [g__Rhizobium, dual-nature]


## 2. Phylogenetic Distance-Based Subclade Clustering

Use `phylogenetic_tree_distance_pairs` to get pairwise distances between
genomes within each target species, then cluster via agglomerative clustering.
This avoids the 1B-row `gene` and `gene_genecluster_junction` tables entirely.

In [3]:
CACHE_SUBCLADE = f'{DATA}/species_subclade_definitions.csv'

if os.path.exists(CACHE_SUBCLADE) and os.path.getsize(CACHE_SUBCLADE) > 100:
    all_subclades = pd.read_csv(CACHE_SUBCLADE)
    print(f'Loaded cached subclade definitions: {len(all_subclades)} genome assignments')
    target_species = all_subclades['gtdb_species_clade_id'].unique().tolist()
else:
    all_subclades = []
    
    for sp_id in target_species:
        print(f'\nProcessing: {sp_id[:60]}...')
        
        # Get pairwise phylogenetic distances via phylogenetic_tree bridge
        # Schema: phylogenetic_tree_distance_pairs has phylogenetic_tree_id, genome1_id, genome2_id, branch_distance
        # phylogenetic_tree has gtdb_species_clade_id -> phylogenetic_tree_id
        phylo_dist = spark.sql(f"""
            SELECT pd.genome1_id, pd.genome2_id, pd.branch_distance
            FROM kbase_ke_pangenome.phylogenetic_tree_distance_pairs pd
            JOIN kbase_ke_pangenome.phylogenetic_tree pt
                 ON pd.phylogenetic_tree_id = pt.phylogenetic_tree_id
            WHERE pt.gtdb_species_clade_id = '{sp_id}'
        """).toPandas()
        
        if len(phylo_dist) == 0:
            print(f'  No phylogenetic distance data, skipping')
            continue
        
        # Get unique genomes
        genomes = sorted(set(phylo_dist['genome1_id']) | set(phylo_dist['genome2_id']))
        n_genomes = len(genomes)
        print(f'  Genomes: {n_genomes}, Pairwise distances: {len(phylo_dist):,}')
        
        if n_genomes < 10:
            print(f'  Too few genomes, skipping')
            continue
        
        # Build distance matrix
        genome_idx = {g: i for i, g in enumerate(genomes)}
        dist_matrix = np.zeros((n_genomes, n_genomes))
        for _, row in phylo_dist.iterrows():
            i = genome_idx.get(row['genome1_id'])
            j = genome_idx.get(row['genome2_id'])
            if i is not None and j is not None:
                dist_matrix[i, j] = row['branch_distance']
                dist_matrix[j, i] = row['branch_distance']
        
        # Agglomerative clustering
        n_clusters = min(3, max(2, n_genomes // 15))
        clustering = AgglomerativeClustering(
            n_clusters=n_clusters, metric='precomputed', linkage='average')
        labels = clustering.fit_predict(dist_matrix)
        
        # MDS for visualization (2D embedding of distance matrix)
        from sklearn.manifold import MDS
        mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42, 
                  max_iter=300, normalized_stress='auto')
        coords = mds.fit_transform(dist_matrix)
        
        sp_subclades = pd.DataFrame({
            'genome_id': genomes,
            'gtdb_species_clade_id': sp_id,
            'subclade': labels,
            'mds1': coords[:, 0],
            'mds2': coords[:, 1],
        })
        
        # Map environment info
        sp_subclades = sp_subclades.merge(
            ge[['genome_id', 'is_plant_associated', 'compartment']], 
            on='genome_id', how='left')
        sp_subclades = sp_subclades.merge(
            host_genomes[['genome_id', 'host_species']].drop_duplicates(),
            on='genome_id', how='left')
        
        all_subclades.append(sp_subclades)
        
        for sc in sorted(sp_subclades['subclade'].unique()):
            sub = sp_subclades[sp_subclades['subclade'] == sc]
            n_plant = sub['is_plant_associated'].fillna(False).sum()
            print(f'  Subclade {sc}: {len(sub)} genomes ({n_plant} plant-associated)')
    
    if all_subclades:
        all_subclades = pd.concat(all_subclades, ignore_index=True)
        all_subclades.to_csv(CACHE_SUBCLADE, index=False)
        print(f'\nSaved: {CACHE_SUBCLADE} ({len(all_subclades)} genome assignments)')
    else:
        all_subclades = pd.DataFrame()
        print('No species had sufficient data for subclade analysis')


Processing: s__Xanthomonas_euvesicatoria--RS_GCF_020880415.1...


  Genomes: 332, Pairwise distances: 54,946


  Subclade 0: 3 genomes (0 plant-associated)
  Subclade 1: 1 genomes (False plant-associated)
  Subclade 2: 328 genomes (0 plant-associated)

Processing: s__Pseudomonas_E_avellanae--RS_GCF_000444135.1...


  Genomes: 319, Pairwise distances: 50,721


  Subclade 0: 16 genomes (0 plant-associated)
  Subclade 1: 125 genomes (0 plant-associated)
  Subclade 2: 178 genomes (0 plant-associated)

Processing: s__Sinorhizobium_meliloti--RS_GCF_017876815.1...


...

  Subclade 0: 234 genomes (0 plant-associated)
  Subclade 1: 1 genomes (False plant-associated)
  Subclade 2: 6 genomes (0 plant-associated)

Processing: s__Pseudomonas_E_amygdali--RS_GCF_002699855.1...


  Genomes: 239, Pairwise distances: 28,441


  Subclade 0: 228 genomes (0 plant-associated)
  Subclade 1: 10 genomes (0 plant-associated)
  Subclade 2: 1 genomes (False plant-associated)

Processing: s__Rhizobium_laguerreae--RS_GCF_002008165.1...


  Genomes: 175, Pairwise distances: 15,225


  Subclade 0: 78 genomes (0 plant-associated)
  Subclade 1: 22 genomes (0 plant-associated)
  Subclade 2: 75 genomes (0 plant-associated)

Saved: ../data/species_subclade_definitions.csv (1306 genome assignments)


## 3. Subclade × Environment Enrichment

Test whether plant-derived genomes cluster into specific subclades
(Fisher's exact test per species).

In [4]:
enrichment_results = []

if len(all_subclades) > 0:
    for sp_id in all_subclades['gtdb_species_clade_id'].unique():
        sp_data = all_subclades[all_subclades['gtdb_species_clade_id'] == sp_id]
        n_subclades = sp_data['subclade'].nunique()
        
        if n_subclades < 2:
            continue
        
        # Overall plant fraction
        total_plant = sp_data['is_plant_associated'].fillna(False).sum()
        total_nonplant = len(sp_data) - total_plant
        
        if total_plant < 3 or total_nonplant < 3:
            enrichment_results.append({
                'species': sp_id,
                'n_genomes': len(sp_data),
                'n_subclades': n_subclades,
                'n_plant': total_plant,
                'n_nonplant': total_nonplant,
                'best_subclade': -1,
                'best_subclade_plant_frac': np.nan,
                'fisher_p': np.nan,
                'chi2_p': np.nan,
                'testable': False,
            })
            continue
        
        # Chi-square test: is plant-association independent of subclade?
        contingency = pd.crosstab(
            sp_data['subclade'], 
            sp_data['is_plant_associated'].fillna(False))
        
        # Ensure both columns exist
        if contingency.shape[1] < 2:
            enrichment_results.append({
                'species': sp_id,
                'n_genomes': len(sp_data),
                'n_subclades': n_subclades,
                'n_plant': total_plant,
                'n_nonplant': total_nonplant,
                'best_subclade': -1,
                'best_subclade_plant_frac': np.nan,
                'fisher_p': np.nan,
                'chi2_p': np.nan,
                'testable': False,
            })
            continue
        
        chi2, chi2_p, _, _ = stats.chi2_contingency(contingency)
        
        # Find the most plant-enriched subclade
        subclade_stats = []
        for sc in sp_data['subclade'].unique():
            sub = sp_data[sp_data['subclade'] == sc]
            n_sc_plant = sub['is_plant_associated'].fillna(False).sum()
            frac = n_sc_plant / len(sub) if len(sub) > 0 else 0
            subclade_stats.append((sc, len(sub), n_sc_plant, frac))
        
        best_sc = max(subclade_stats, key=lambda x: x[3])
        
        # Fisher's exact for the most enriched subclade vs rest
        a = best_sc[2]  # plant in best subclade
        b = best_sc[1] - best_sc[2]  # non-plant in best subclade
        c = total_plant - a  # plant in other subclades
        d = total_nonplant - b  # non-plant in other subclades
        _, fisher_p = stats.fisher_exact([[a, b], [c, d]])
        
        enrichment_results.append({
            'species': sp_id,
            'n_genomes': len(sp_data),
            'n_subclades': n_subclades,
            'n_plant': total_plant,
            'n_nonplant': total_nonplant,
            'best_subclade': best_sc[0],
            'best_subclade_plant_frac': best_sc[3],
            'fisher_p': fisher_p,
            'chi2_p': chi2_p,
            'testable': True,
        })

enrichment_df = pd.DataFrame(enrichment_results)
if len(enrichment_df) > 0:
    print('=== Subclade × Plant-Association Enrichment ===')
    print(f'{"Species":<55} {"N":>4} {"SC":>3} {"Plant":>5} {"Best%":>6} {"Fisher":>10} {"Chi2":>10}')
    for _, row in enrichment_df.sort_values('fisher_p').iterrows():
        sp_short = row['species'][:54]
        print(f'{sp_short:<55} {row["n_genomes"]:>4} {row["n_subclades"]:>3} '
              f'{row["n_plant"]:>5.0f} {row["best_subclade_plant_frac"]:>5.0%} '
              f'{row["fisher_p"]:>10.2e} {row["chi2_p"]:>10.2e}')
    
    sig = enrichment_df[(enrichment_df['testable']) & (enrichment_df['chi2_p'] < 0.05)]
    print(f'\nSpecies with significant subclade × plant enrichment (chi2 p<0.05): {len(sig)}')
else:
    print('No enrichment results to report')

=== Subclade × Plant-Association Enrichment ===
Species                                                    N  SC Plant  Best%     Fisher       Chi2
s__Xanthomonas_euvesicatoria--RS_GCF_020880415.1         332   3     0  nan%        nan        nan
s__Pseudomonas_E_avellanae--RS_GCF_000444135.1           319   3     0  nan%        nan        nan
s__Sinorhizobium_meliloti--RS_GCF_017876815.1            241   3     0  nan%        nan        nan
s__Pseudomonas_E_amygdali--RS_GCF_002699855.1            239   3     0  nan%        nan        nan
s__Rhizobium_laguerreae--RS_GCF_002008165.1              175   3     0  nan%        nan        nan

Species with significant subclade × plant enrichment (chi2 p<0.05): 0


## 4. Subclade × Host Plant Mapping

For species with host plant annotations, check whether specific hosts
map to specific subclades.

In [5]:
host_mapping = []

if len(all_subclades) > 0:
    for sp_id in all_subclades['gtdb_species_clade_id'].unique():
        sp_data = all_subclades[all_subclades['gtdb_species_clade_id'] == sp_id]
        with_host = sp_data[sp_data['host_species'].notna()]
        
        if len(with_host) < 5 or with_host['host_species'].nunique() < 2:
            continue
        
        # Subclade × host contingency
        ct = pd.crosstab(with_host['subclade'], with_host['host_species'])
        if ct.shape[0] >= 2 and ct.shape[1] >= 2:
            chi2, p, _, _ = stats.chi2_contingency(ct)
            host_mapping.append({
                'species': sp_id,
                'n_with_host': len(with_host),
                'n_hosts': with_host['host_species'].nunique(),
                'chi2_p': p,
                'top_host_per_subclade': {sc: with_host[with_host['subclade'] == sc]['host_species']
                    .value_counts().index[0] if len(with_host[with_host['subclade'] == sc]) > 0 else 'NA'
                    for sc in sorted(sp_data['subclade'].unique())}
            })

host_df = pd.DataFrame(host_mapping)
if len(host_df) > 0:
    print('=== Subclade × Host Plant Associations ===')
    for _, row in host_df.sort_values('chi2_p').iterrows():
        sig = '*' if row['chi2_p'] < 0.05 else ''
        print(f'{row["species"][:55]}: {row["n_with_host"]} genomes, '
              f'{row["n_hosts"]} hosts, chi2 p={row["chi2_p"]:.3e} {sig}')
        for sc, host in row['top_host_per_subclade'].items():
            print(f'    Subclade {sc}: dominant host = {host}')
    
    host_df.to_csv(f'{DATA}/subclade_host_mapping.csv', index=False)
    print(f'\nSaved: {DATA}/subclade_host_mapping.csv')
else:
    print('No species had sufficient host data for subclade × host analysis')
    pd.DataFrame().to_csv(f'{DATA}/subclade_host_mapping.csv', index=False)

No species had sufficient host data for subclade × host analysis


In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if len(all_subclades) > 0:
    species_list = all_subclades['gtdb_species_clade_id'].unique()
    n_species = min(len(species_list), 5)
    
    fig, axes = plt.subplots(1, n_species, figsize=(5 * n_species, 5))
    if n_species == 1:
        axes = [axes]
    
    for i, sp_id in enumerate(species_list[:n_species]):
        sp_data = all_subclades[all_subclades['gtdb_species_clade_id'] == sp_id]
        
        for sc in sorted(sp_data['subclade'].unique()):
            sc_data = sp_data[sp_data['subclade'] == sc]
            plant = sc_data[sc_data['is_plant_associated'] == True]
            nonplant = sc_data[sc_data['is_plant_associated'] != True]
            
            markers = ['o', 's', '^', 'D', 'v'][sc % 5]
            if len(plant) > 0:
                axes[i].scatter(plant['mds1'], plant['mds2'], 
                               marker=markers, c='forestgreen', alpha=0.6, s=30)
            if len(nonplant) > 0:
                axes[i].scatter(nonplant['mds1'], nonplant['mds2'],
                               marker=markers, c='gray', alpha=0.4, s=30)
        
        genus = sp_comp[sp_comp['gtdb_species_clade_id'] == sp_id]['genus'].iloc[0]
        n_plant = sp_data['is_plant_associated'].fillna(False).sum()
        axes[i].set_title(f'{genus}\n{len(sp_data)} genomes ({n_plant} plant)', fontsize=9)
        axes[i].set_xlabel('MDS1')
        if i == 0:
            axes[i].set_ylabel('MDS2')
    
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='forestgreen', markersize=8, label='Plant'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=8, label='Non-plant'),
    ]
    axes[-1].legend(handles=legend_elements, loc='lower right', fontsize=8)
    
    plt.tight_layout()
    plt.savefig(f'{FIG}/subclade_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {FIG}/subclade_analysis.png')
else:
    print('No subclade data to plot')

Saved: ../figures/subclade_analysis.png


In [7]:
# Save enrichment results
if len(enrichment_df) > 0:
    enrichment_df.to_csv(f'{DATA}/subclade_og_enrichment.csv', index=False)
    print(f'Saved: {DATA}/subclade_og_enrichment.csv')
else:
    pd.DataFrame().to_csv(f'{DATA}/subclade_og_enrichment.csv', index=False)

print('\n' + '='*80)
print('NB12 SUMMARY: Within-Species Subclade OG Analysis')
print('='*80)

if len(all_subclades) > 0:
    print(f'\nSpecies analyzed: {all_subclades["gtdb_species_clade_id"].nunique()}')
    print(f'Total genome assignments: {len(all_subclades)}')
    
    if len(enrichment_df) > 0:
        testable = enrichment_df[enrichment_df['testable']]
        sig = testable[testable['chi2_p'] < 0.05]
        print(f'Testable species: {len(testable)}')
        print(f'Significant subclade × plant enrichment: {len(sig)}')
    
    if len(host_df) > 0:
        host_sig = host_df[host_df['chi2_p'] < 0.05]
        print(f'Species with subclade × host association: {len(host_sig)}')
else:
    print('No subclade analysis was possible')

print(f'\nOutputs:')
print(f'  data/species_subclade_definitions.csv')
print(f'  data/subclade_og_enrichment.csv')
print(f'  data/subclade_host_mapping.csv')
print(f'  figures/subclade_analysis.png')

Saved: ../data/subclade_og_enrichment.csv

NB12 SUMMARY: Within-Species Subclade OG Analysis

Species analyzed: 5
Total genome assignments: 1306
Testable species: 0
Significant subclade × plant enrichment: 0

Outputs:
  data/species_subclade_definitions.csv
  data/subclade_og_enrichment.csv
  data/subclade_host_mapping.csv
  figures/subclade_analysis.png
